In [2]:
import pandas as pd
import numpy as np
import sys, json, joblib, warnings

from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_validate, train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore', category=UserWarning)

In [10]:
root = '/content/drive/MyDrive/Portnet Imputation Prediction'
pipeline = root + "/src"

raw_data = pd.read_csv(f'{root}/data.dsv', sep=';')
if pipeline not in sys.path: sys.path.append(pipeline)

from data_pipeline import portnet_pipeline
from google.colab import drive

drive.mount('/content/drive')

# HuggingFace Config

In [ ]:
from huggingface_hub import notebook_login
from huggingface_hub import HfApi, create_repo

notebook_login()

In [5]:
repo_id = "Meliodas-10/portnet-model"
create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

RepoUrl('https://huggingface.co/Meliodas-10/portnet-model', endpoint='https://huggingface.co', repo_type='model', repo_id='Meliodas-10/portnet-model')

# Dataset Config

In [15]:
df = raw_data[raw_data['QTE_IMPUTE'] > 0].copy()
df = df[df['DEVISE'].isin(['EUR', 'USD'])]

X = df.drop(columns=['QTE_IMPUTE'])
y = df['QTE_IMPUTE']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


X_train_sample = X_train.sample(n=200000, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]

# 1- Algorithm Selection and Training

## 1.1- Dummy Model

In [10]:
from sklearn.dummy import DummyRegressor

model = TransformedTargetRegressor(
    regressor=DummyRegressor(strategy='mean'),
    func=np.log1p,
    inverse_func=np.expm1
)

model.fit(X_train_sample, y_train_sample)

y_pred_baseline = model.predict(X_test)

mae_base = mean_absolute_error(y_test, y_pred_baseline)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
r2_base = r2_score(y_test, y_pred_baseline)

print("--- BASELINE Score ---")
print(f"MAE  : {mae_base:.2f}")
print(f"RMSE : {rmse_base:.2f}")
print(f"R²   : {r2_base:.4f}")

--- BASELINE Score ---
MAE  : 113029.09
RMSE : 2484896.59
R²   : -0.0020


## 1.2- Models CV Loop

In [11]:
algorithmes = {
    "Random Forest (Bagging)": RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    "HistGradientBoosting (Boosting)": HistGradientBoostingRegressor(random_state=42)
}

In [12]:
test_slice = X_train.head(5)
target_slice = y_train.head(5)

print("Testing pipeline on a small slice...")
modele_complet = TransformedTargetRegressor(
    regressor=portnet_pipeline.set_params(algorithme=algorithmes["Random Forest (Bagging)"]),
    func=np.log1p,
    inverse_func=np.expm1
)
modele_complet.fit(test_slice, target_slice)
print("\n----- Sanity check passed successfully! The pipeline fits without errors -----")

Testing pipeline on a small slice...
[_drop_unnecessary_columns] exécuté en 0.001s | Lignes restantes: 5
[_clean_quantite_domicile] exécuté en 0.001s | Lignes restantes: 5
[_set_datatypes] exécuté en 0.008s | Lignes restantes: 5
[_optimize_memory] exécuté en 0.003s | Lignes restantes: 5
[_delai_extracting] exécuté en 0.002s | Lignes restantes: 5
[_strategic_grouping] exécuté en 0.001s | Lignes restantes: 5
[_temporal_engineering] exécuté en 0.002s | Lignes restantes: 5
[_flags_creation] exécuté en 0.001s | Lignes restantes: 5
[_unify_currency_to_eur] exécuté en 0.005s | Lignes restantes: 5
[_logarithmic_transform] exécuté en 0.001s | Lignes restantes: 5
[_impute_and_scale] exécuté en 0.002s | Lignes restantes: 5
[_drop_useless_text] exécuté en 0.000s | Lignes restantes: 5
[_target_encode] exécuté en 0.001s | Lignes restantes: 5

----- Sanity check passed successfully! The pipeline fits without errors -----


In [ ]:
X_train_sample = X_train.sample(n=200000, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]

for nom, algo in algorithmes.items():
    print(f"\n--- Entraînement de {nom} en cours (CV = 5) ---")
    portnet_pipeline.set_params(algorithme=algo)

    modele_complet = TransformedTargetRegressor(
        regressor=portnet_pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )

    scores = cross_validate(
        modele_complet, 
        X_train_sample, 
        y_train_sample, 
        cv=5, 
        scoring=('neg_mean_absolute_error', 'r2'),
        n_jobs=1
    )
 
    mae_moyen = -scores['test_neg_mean_absolute_error'].mean()
    r2_moyen = scores['test_r2'].mean()
    
    print(f"[{nom}] R² Moyen  : {r2_moyen:.4f}")
    print(f"[{nom}] MAE Moyen : {mae_moyen:.2f} unités")

## Results:
Random Forest (Bagging):
* R² Moyen  : 0.6608
* MAE Moyen : 35725.30 unités


HistGradientBoosting (Boosting):
* R² Moyen  : 0.5146
* MAE Moyen : 53058.07 unités

### --> Decision : Random Forest

# 2- Hyperparameter Tuning

In [ ]:
portnet_pipeline.set_params(algorithme=RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))

model = TransformedTargetRegressor(
    regressor=portnet_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

param_distributions = {
    'regressor__algorithme__n_estimators': [100, 200, 300],
    'regressor__algorithme__max_depth': [15, 25, None],
    'regressor__algorithme__min_samples_split': [2, 5, 10],
    'regressor__algorithme__min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=5,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=1
)

print("--- Début de l'optimisation des hyperparamètres ---")
random_search.fit(X_train_sample, y_train_sample)

print(f"\nMeilleurs paramètres trouvés : {random_search.best_params_}")
print(f"Meilleur score R² (CV) : {random_search.best_score_:.4f}")

best_params = {k.replace('regressor__', ''): v for k, v in random_search.best_params_.items()}
portnet_pipeline.set_params(**best_params)

with open("best_params.json", "w") as f:
    json.dump(best_params, f)

api = HfApi()
api.upload_file(
    path_or_fileobj="best_params.json",
    path_in_repo="best_params.json",
    repo_id=repo_id,
    repo_type="model"
)

print("Paramètres sauvegardés avec succès sur Hugging Face !")

# 3- Model Execution

In [20]:
for param, value in portnet_pipeline.named_steps['algorithme'].get_params().items():
    print(f"{param}: {value}")

bootstrap: True
ccp_alpha: 0.0
criterion: squared_error
max_depth: 25
max_features: 1.0
max_leaf_nodes: None
max_samples: None
min_impurity_decrease: 0.0
min_samples_leaf: 1
min_samples_split: 5
min_weight_fraction_leaf: 0.0
monotonic_cst: None
n_estimators: 100
n_jobs: -1
oob_score: False
random_state: 42
verbose: 0
warm_start: False


In [21]:
full_model = TransformedTargetRegressor(
    regressor=portnet_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

print("\n--- Entraînement du modèle final sur l'intégralité du dataset ---")
full_model.fit(X_train, y_train)


--- Entraînement du modèle final sur l'intégralité du dataset ---
[_drop_unnecessary_columns] exécuté en 0.300s | Lignes restantes: 3691884
[_clean_quantite_domicile] exécuté en 0.030s | Lignes restantes: 3691884
[_set_datatypes] exécuté en 304.535s | Lignes restantes: 3691884
[_optimize_memory] exécuté en 0.254s | Lignes restantes: 3691884
[_delai_extracting] exécuté en 0.196s | Lignes restantes: 3691884
[_strategic_grouping] exécuté en 0.024s | Lignes restantes: 3691884
[_temporal_engineering] exécuté en 0.213s | Lignes restantes: 3691884
[_flags_creation] exécuté en 0.015s | Lignes restantes: 3691884


/content/drive/MyDrive/Portnet Imputation Prediction/src/data_pipeline.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[    0.   3916.8     0.  ...  3187.5  1249.5 14280. ]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df.loc[mask_usd, cols_to_convert] = df.loc[mask_usd, cols_to_convert] * self.taux_conversion
/content/drive/MyDrive/Portnet Imputation Prediction/src/data_pipeline.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 23730.92919922  75072.         249718.95       ...   8712.67016602
  15733.5        150960.        ]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df.loc[mask_usd, cols_to_convert] = df.loc[mask_usd, cols_to_convert] * self.taux_conversion
/content/drive/MyDrive/Portnet Imputation Prediction/src/

[_unify_currency_to_eur] exécuté en 1.170s | Lignes restantes: 3691884
[_logarithmic_transform] exécuté en 0.060s | Lignes restantes: 3691884
[_impute_and_scale] exécuté en 0.274s | Lignes restantes: 3691884
[_drop_useless_text] exécuté en 0.119s | Lignes restantes: 3691884
[_target_encode] exécuté en 0.558s | Lignes restantes: 3691884


TransformedTargetRegressor(func=<ufunc 'log1p'>, inverse_func=<ufunc 'expm1'>,
                           regressor=Pipeline(steps=[('cleaner',
                                                      PortNetDataCleaner()),
                                                     ('feature_engineer',
                                                      PortNetFeatureEngineering()),
                                                     ('preprocessor',
                                                      PortNetDataPreprocessing()),
                                                     ('algorithme',
                                                      RandomForestRegressor(max_depth=25,
                                                                            min_samples_split=5,
                                                                            n_jobs=-1,
                                                                            random_state=42))]))

In [22]:
y_pred = full_model.predict(X_test)

mae_base = mean_absolute_error(y_test, y_pred)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred))
r2_base = r2_score(y_test, y_pred)

print("--- Final Score ---")
print(f"MAE  : {mae_base:.2f}")
print(f"RMSE : {rmse_base:.2f}")
print(f"R²   : {r2_base:.4f}")

[_drop_unnecessary_columns] exécuté en 0.060s | Lignes restantes: 922972
[_clean_quantite_domicile] exécuté en 0.005s | Lignes restantes: 922972
[_set_datatypes] exécuté en 73.534s | Lignes restantes: 922972
[_optimize_memory] exécuté en 0.040s | Lignes restantes: 922972
[_delai_extracting] exécuté en 0.041s | Lignes restantes: 922972
[_strategic_grouping] exécuté en 0.005s | Lignes restantes: 922972
[_temporal_engineering] exécuté en 0.044s | Lignes restantes: 922972
[_flags_creation] exécuté en 0.002s | Lignes restantes: 922972
[_unify_currency_to_eur] exécuté en 0.145s | Lignes restantes: 922972
[_logarithmic_transform] exécuté en 0.009s | Lignes restantes: 922972
[_impute_and_scale] exécuté en 0.034s | Lignes restantes: 922972
[_drop_useless_text] exécuté en 0.016s | Lignes restantes: 922972
[_target_encode] exécuté en 0.180s | Lignes restantes: 922972
--- Final Score ---
MAE  : 22984.15
RMSE : 758297.75
R²   : 0.8666


In [24]:
joblib.dump(full_model, "portnet_model.pkl")
api = HfApi()

api.upload_file(
    path_or_fileobj="portnet_model.pkl",
    path_in_repo="portnet_model.pkl",
    repo_id=repo_id,
    repo_type="model"
)
print("modèle sauvegardés avec succès sur Hugging Face !")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/portnet_model.pkl  :   0%|          | 1.11MB / 3.78GB            

modèle sauvegardés avec succès sur Hugging Face !
